In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Users/jamesgroombridge@gmail.com/data_platform_ess2_prototype/ess0_to_ess1')
if 'utilities.utility_logging' in sys.modules:
    del sys.modules['utilities.utility_logging']
from utilities.utility_logging import logger
from utilities.utility_datacontract import data_contract_list
from datetime import datetime, timezone

# notebook run
# check all new files are avialble for pipeline to run - completeness
# check files are appropriately sized - validity
# check there is not duplicated files - uniqueness
# log events to volume
# log metrics to volume
# trigger pipeline to run if criteria met

# start logger
logger_instance = logger("pipeline_run")

def file_exists(data_contract_list):
    try:
        logger_instance.info("start file check")
        current_time = datetime.now(timezone.utc)
        
        for schema in data_contract_list:
            name = schema['name']
            files = dbutils.fs.ls(schema['volume'])
            if len(files) == 0:
                logger_instance.info(f"No files found for {name}")
                raise Exception("No files found - stopping pipeline")
            
            logger_instance.info(f"{name} - {len(files)} files found")
            
            # Display files with age in days and size in MB
            file_info = []
            for file in files:
                mod_time = datetime.fromtimestamp(file.modificationTime / 1000, tz=timezone.utc)
                age_seconds = (current_time - mod_time).total_seconds()
                age_days = int(age_seconds / 86400)  # 86400 seconds in a day
                size_kb = round(file.size / (1024), 2)  # Convert bytes to KB
                
                file_info.append({
                    'name': file.name,
                    'size_kb': size_kb,
                    'modified_time': mod_time.strftime('%Y-%m-%d %H:%M:%S UTC'),
                    'age_days': age_days
                })
                if size_kb < 2:
                    logger_instance.info(f"size of file is less than 2kb for {name}")
                    raise Exception("file less than 2kb found - stopping pipeline")
                if age_days > 2:
                    logger_instance.info(f"file age is greater than 7days for {name}")
                    raise Exception("file age is greater than 7days - stopping pipeline")
                
            display(file_info)
            
            # Check for duplicate files
            file_names = [file.name for file in files]
            if len(file_names) != len(set(file_names)):
                logger_instance.info("files are duplicted for {name}")
                raise Exception("files are duplicated - stopping pipeline")
        
        logger_instance.info("end file check")
        return True
    except Exception as e:
        logger_instance.error(f"Error: {e}")

file_exists(data_contract_list())